# AI-Face Phase E — Global / Local baseline

Notebook này chỉ điều phối runner và hiển thị artifact. Logic train/evaluation nằm trong `scripts/run_aiface_phase_e.py`, nên chạy local và Kaggle dùng cùng một protocol.

Phần này không chạy Residual. Dataset Kaggle mặc định là `nguyentrann0703/aiface-pilot-32k-single-bin`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/FaceTrace.git'
REPO_DIR = Path('/kaggle/working/FaceTrace')
RUN_SMOKE = False  # Đổi thành True để preflight trước full run
OUTPUT_DIR = 'outputs/aiface_phase_e'

if not (REPO_DIR / 'scripts/run_aiface_phase_e_kaggle.py').is_file():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
print('Repo:', REPO_DIR)
print('Dataset: nguyentrann0703/aiface-pilot-32k-single-bin')

In [ ]:
import os

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError('Kaggle Secret HF_TOKEN chưa được gắn cho kernel này.') from exc
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    del HF_TOKEN
print('Hugging Face authentication: ready')

In [ ]:
command = [
    sys.executable,
    str(REPO_DIR / 'scripts/run_aiface_phase_e_kaggle.py'),
    '--config', str(REPO_DIR / 'configs/aiface_phase_e.yaml'),
    '--output-dir', OUTPUT_DIR,
]
if RUN_SMOKE:
    command.append('--smoke')
subprocess.run(command, cwd=REPO_DIR, check=True)

In [ ]:
import json
import pandas as pd
from IPython.display import display

output = REPO_DIR / OUTPUT_DIR
for name in ['global_only', 'local_only', 'global_local']:
    metrics = json.loads((output / name / 'test_metrics_overall.json').read_text())
    print(name)
    display(pd.DataFrame([metrics]))
    display(pd.read_csv(output / name / 'test_metrics_by_generator.csv'))

print('Shortcut diagnostic')
display(pd.read_csv(output / 'shortcut' / 'metrics.csv'))
print('Metadata')
display(json.loads((output / 'run_metadata.json').read_text()))